# Othello — Colab

Auteur : **Théo Cornu** · Contrat : **v1.1**

## Préparation
Déposer dans `/content` :
- `config.py`, `players_config.json`
- `engine.py`, `evals.py`, `agent.py`, `human.py`, `players.py`, `harness.py`, `cli.py`, `tests.py`
- le dossier `web/` (`server.py`, `index.html`)


In [ ]:
import sys, os
sys.path.insert(0, '/content')
from config import CONTRACT_VERSION
from engine import OthelloEngine
from evals import make_evaluator, EVALUATION_NAMES
from agent import Agent, MoveResult
from human import HumanPlayer
from harness import play_game, GameResult
from players import load_player_config, make_profile_player
ENGINE = OthelloEngine()
CONFIG = load_player_config()
print('Contrat', CONTRACT_VERSION)
print('Profils', list(CONFIG['profiles'].keys()))
print('Évaluations', EVALUATION_NAMES)

## 1. Tests de conformité
- `test_perft` : 4, 12, 56, 244, 1396, 8200
- `test_symmetry_and_bounds` : `eval(s, 1) == -eval(s, 2)` et `|eval| < 1000`
- `test_hash_consistency` : hash Zobrist recalculé

In [ ]:
from tests import test_perft, test_symmetry_and_bounds, test_hash_consistency
test_perft()
test_symmetry_and_bounds()
test_hash_consistency()

## 2. IA contre IA
Deux profils de `players_config.json`, deux caches distincts.

In [ ]:
ia_noire = make_profile_player(ENGINE, 'mixed_ai', CONFIG)
ia_blanche = make_profile_player(ENGINE, 'positional_ai', CONFIG)
resultat = play_game(ENGINE, ia_noire, ia_blanche, game_id='colab_ia_vs_ia')
print('Gagnant', resultat.winner, '| score', resultat.score, '| coups', len(resultat.moves))
print('Dernier MoveResult :', resultat.move_results[-1])

## 3. Humain contre IA
Le joueur humain saisit un coup en notation `d3` ou `pass`.

In [ ]:
ia = make_profile_player(ENGINE, 'mixed_ai', CONFIG)
humain = make_profile_player(ENGINE, 'human', CONFIG)
resultat = play_game(ENGINE, ia, humain, game_id='colab_humain_vs_ia')
print('Score final', resultat.score, '| gagnant', resultat.winner)

## 4. Campagne de matchs
Boucle de parties entre stratégies, résultats numériques.

In [ ]:
def campagne(profil_noir, profil_blanc, nb_parties=5):
    lignes = []
    for index in range(nb_parties):
        noir = make_profile_player(ENGINE, profil_noir, CONFIG)
        blanc = make_profile_player(ENGINE, profil_blanc, CONFIG)
        resultat = play_game(ENGINE, noir, blanc, game_id=f'{profil_noir}_vs_{profil_blanc}_{index}')
        lignes.append((index, resultat.winner, resultat.score[0], resultat.score[1]))
    return lignes

for ligne in campagne('mixed_ai', 'mobility_ai', 3):
    print(ligne)

## 5. Interface web dans Colab
Lance le serveur `web/server.py` puis affiche l'interface dans un iframe Colab.
Le bouton **Pause** arrête l'auto-play à tout moment.

In [ ]:
import threading, socketserver, importlib.util, time

spec = importlib.util.spec_from_file_location('othello_web_server', '/content/web/server.py')
serveur = importlib.util.module_from_spec(spec)
spec.loader.exec_module(serveur)

PORT = 8000
def lancer_serveur():
    with socketserver.TCPServer(('', PORT), serveur.RequestHandler) as httpd:
        httpd.serve_forever()

threading.Thread(target=lancer_serveur, daemon=True).start()
time.sleep(1)
print(f'Serveur prêt sur le port {PORT}')

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=820)
except Exception:
    print('Hors Colab : ouvrir http://localhost:8000 dans le navigateur.')